# Apache Spark Data Lake Preprocessing
### Data Mining Assignment
**Dataset**: `sales.csv` | **Environment**: Google Colab + PySpark | **Layers**: Bronze (Raw) & Silver (Cleaned)

---
### Architecture Overview
This notebook implements a two-tier Data Lake architecture using Apache Spark:
1. **Bronze Layer (`data_lake/bronze/`)**: Stores raw, immutable data (`sales.csv`) exactly as received.
2. **Silver Layer (`data_lake/silver/preprocessed_sales/`)**: Stores preprocessed, validated, deduplicated, and standardized data.

No Gold layer or detailed data analysis is performed as per the assignment specifications.


In [1]:
# Install PySpark in Google Colab environment
!pip install -q pyspark


In [2]:
import os
import sys
import shutil

# 1. Update packages and install OpenJDK 17 to match Python 3.13's expectations
print("Updating package lists and installing openjdk-17-jdk...")
!apt-get update -qq && apt-get install -y openjdk-17-jdk-headless -qq > /dev/null

# 2. Configure Environment variables to point to the newly installed Java 17 JVM
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

# Ensure Spark uses Colab's Python environment
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

print("Java found at:", shutil.which("java"))

# --- Imports ---
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType, DateType
)
from pyspark.sql.window import Window
from datetime import datetime

# --- Initialize SparkSession ---
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Sales_Data_Lake_Preprocessing")
    .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .getOrCreate()
)

print(f"Spark Version: {spark.version}")

Updating package lists and installing openjdk-17-jdk...
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Java found at: /usr/lib/jvm/java-17-openjdk-amd64/bin/java
Spark Version: 4.0.5


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1. Data Lake Structure
Create the following folders in the Colab runtime:
```
data_lake/
|-- bronze/
|   `-- sales.csv
`-- silver/
    `-- preprocessed_sales/
```
The Bronze file must remain exactly as provided without any modifications.


In [4]:
# Create Data Lake directory structure
bronze_dir = "data_lake/bronze"
silver_dir = "data_lake/silver/preprocessed_sales"
export_silver_dir = "silver"

os.makedirs(bronze_dir, exist_ok=True)
os.makedirs(silver_dir, exist_ok=True)
os.makedirs(export_silver_dir, exist_ok=True)

# Place sales.csv in the Bronze layer if not already present
bronze_file = os.path.join(bronze_dir, "sales.csv")
if not os.path.exists(bronze_file):
    candidate_files = ["sales.csv", "sales (3).csv"]
    copied = False
    for candidate in candidate_files:
        if os.path.exists(candidate):
            shutil.copy(candidate, bronze_file)
            print(f"Copied {candidate} -> {bronze_file}")
            copied = True
            break
    if not copied:
        print(f"Please ensure sales.csv is uploaded to {bronze_dir}/")
else:
    print(f"Bronze file already present at: {bronze_file}")

# Verify Bronze directory contents
print("\nFiles in Bronze Layer:")
for f in os.listdir(bronze_dir):
    size = os.path.getsize(os.path.join(bronze_dir, f))
    print(f"  - {f} ({size:,} bytes)")


Please ensure sales.csv is uploaded to data_lake/bronze/

Files in Bronze Layer:


## 2. Load and Inspect with Spark
- Read `sales.csv` from Bronze.
- Display the schema, sample rows, and the total record count.
- Use Spark operations to inspect data quality issues:
  - Exact duplicates
  - Null / blank values
  - Text-format inconsistencies (whitespace, casing)
  - Invalid numerical values
  - Inconsistent / invalid dates


In [6]:
# Read sales.csv from Bronze layer
raw_df = spark.read.csv(
    bronze_file,
    header=True,
    inferSchema=True
)

# Display schema
print("=== Raw DataFrame Schema (Bronze Layer) ===")
raw_df.printSchema()

# Display sample records
print("=== First 5 Records ===")
raw_df.show(5, truncate=False)

# Total record count
total_raw_count = raw_df.count()
print(f"Total Record Count in Bronze Layer: {total_raw_count:,}")


=== Raw DataFrame Schema (Bronze Layer) ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

=== First 5 Records ===
+--------+-----------+--------------+-------------+--------+--------+----------+----------------+--------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name |product      |category|quantity|unit_price|discount_percent|payment_method|city      |state         |order_date|order_status|
+--------+-----------+--------------+-------------+---

### 2.1 Inspection: Exact Duplicate Rows
Identify identical rows across all columns using Spark aggregation operations.


In [7]:
# Calculate unique records and duplicate counts using Spark operations
distinct_raw_count = raw_df.distinct().count()
duplicate_count = total_raw_count - distinct_raw_count
print(f"Total Rows     : {total_raw_count}")
print(f"Distinct Rows  : {distinct_raw_count}")
print(f"Duplicate Rows : {duplicate_count}")

# Display sample of duplicate rows
print("\nSample of duplicate rows found in dataset:")
raw_df.groupBy(raw_df.columns) \
    .count() \
    .filter(F.col("count") > 1) \
    .show(5, truncate=False)


Total Rows     : 1000
Distinct Rows  : 990
Duplicate Rows : 10

Sample of duplicate rows found in dataset:
+--------+-----------+--------------+---------------+---------------+--------+----------+----------------+----------------+----------+-------------+----------+------------+-----+
|order_id|customer_id|customer_name |product        |category       |quantity|unit_price|discount_percent|payment_method  |city      |state        |order_date|order_status|count|
+--------+-----------+--------------+---------------+---------------+--------+----------+----------------+----------------+----------+-------------+----------+------------+-----+
|100692  |C0141      |Varun Agarwal |Electric Kettle|Home Appliances|4       |920       |0               |Net Banking     |Delhi     |Delhi        |2026-08-18|Shipped     |2    |
|100145  |C0007      |Sai Mishra    |Face Wash      |Beauty         |7       |350       |30              |Cash on Delivery|Bengaluru |Karnataka    |2025-08-28|Delivered   |2    

### 2.2 Inspection: Missing / Null / Blank Values
Inspect each column for `NULL`, `NaN`, or empty/whitespace-only strings using Spark aggregation.


In [9]:
# Inspect missing/null/blank counts per column using Spark safely
from pyspark.sql.types import DoubleType, FloatType

# Helper to determine if a column is float/double type
def is_nan_applicable(col_name):
    dtype = dict(raw_df.dtypes)[col_name]
    return dtype in ("double", "float")

null_counts = raw_df.select([
    F.count(
        F.when(
            F.col(c).isNull() |
            (F.isnan(F.col(c)) if is_nan_applicable(c) else F.lit(False)) |
            (F.trim(F.col(c).cast("string")) == ""),
            c
        )
    ).alias(c)
    for c in raw_df.columns
])

print("=== Missing / Null / Blank Values per Column ===")
null_counts.show(vertical=True)

=== Missing / Null / Blank Values per Column ===
-RECORD 0---------------
 order_id         | 0   
 customer_id      | 0   
 customer_name    | 10  
 product          | 0   
 category         | 0   
 quantity         | 0   
 unit_price       | 0   
 discount_percent | 0   
 payment_method   | 8   
 city             | 8   
 state            | 0   
 order_date       | 0   
 order_status     | 0   



### 2.3 Inspection: Text Formatting Inconsistencies
Identify leading/trailing whitespaces and casing discrepancies in string columns (`customer_name`, `category`, `city`, `payment_method`, etc.).


In [10]:
# Check for leading/trailing whitespace across text columns
text_cols = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]

print("=== Columns with Leading / Trailing Whitespace ===")
for c in text_cols:
    ws_count = raw_df.filter(F.col(c).isNotNull() & (F.col(c) != F.trim(F.col(c)))).count()
    print(f"  {c:15s}: {ws_count} rows with extra whitespace")

# Check casing variations in 'category' and 'city'
print("\n=== Distinct values in 'category' (Notice mixed casing like SPORTS vs Sports) ===")
raw_df.select("category").distinct().show(truncate=False)

print("=== Distinct values in 'city' (Notice mixed casing like DELHI, delhi, Delhi and whitespace) ===")
raw_df.select("city").distinct().show(25, truncate=False)


=== Columns with Leading / Trailing Whitespace ===
  customer_name  : 3 rows with extra whitespace
  product        : 0 rows with extra whitespace
  category       : 0 rows with extra whitespace
  payment_method : 0 rows with extra whitespace
  city           : 3 rows with extra whitespace
  state          : 0 rows with extra whitespace
  order_status   : 0 rows with extra whitespace

=== Distinct values in 'category' (Notice mixed casing like SPORTS vs Sports) ===
+---------------+
|category       |
+---------------+
|SPORTS         |
|beauty         |
|Sports         |
|Grocery        |
|Electronics    |
|Clothing       |
|Books          |
|ELECTRONICS    |
|Home Appliances|
|Furniture      |
|sports         |
|Beauty         |
|CLOTHING       |
+---------------+

=== Distinct values in 'city' (Notice mixed casing like DELHI, delhi, Delhi and whitespace) ===
+-----------+
|city       |
+-----------+
|delhi      |
| Hyderabad |
|Chennai    |
|Lucknow    |
|Mumbai     |
|DELHI      |
|

### 2.4 Inspection: Invalid Numerical Values
Check for clearly invalid numbers:
- `quantity <= 0`
- `unit_price <= 0`
- `discount_percent < 0` or `discount_percent > 100`


In [11]:
# Identify invalid numerical values using Spark filters
invalid_qty_df = raw_df.filter((F.col("quantity") <= 0) | F.col("quantity").isNull())
invalid_price_df = raw_df.filter((F.col("unit_price") <= 0) | F.col("unit_price").isNull())
invalid_disc_df = raw_df.filter(
    (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100) | F.col("discount_percent").isNull()
)

print(f"Records with invalid quantity (<= 0): {invalid_qty_df.count()}")
invalid_qty_df.select("order_id", "product", "quantity").show(5)

print(f"Records with invalid unit_price (<= 0): {invalid_price_df.count()}")
invalid_price_df.select("order_id", "product", "unit_price").show(5)

print(f"Records with invalid discount_percent (< 0 or > 100): {invalid_disc_df.count()}")
invalid_disc_df.select("order_id", "product", "discount_percent").show(5)


Records with invalid quantity (<= 0): 12
+--------+----------------+--------+
|order_id|         product|quantity|
+--------+----------------+--------+
|  100114|     Ceiling Fan|       0|
|  100143|        Football|       0|
|  100198|          Hoodie|      -2|
|  100365|  Wireless Mouse|       0|
|  100375|Basmati Rice 5kg|       0|
+--------+----------------+--------+
only showing top 5 rows
Records with invalid unit_price (<= 0): 8
+--------+--------------------+----------+
|order_id|             product|unit_price|
+--------+--------------------+----------+
|  100102|           Atta 10kg|         0|
|  100170|Competitive Exam ...|         0|
|  100235|          Power Bank|      -500|
|  100284|         Study Table|      -500|
|  100487|        Office Chair|      -500|
+--------+--------------------+----------+
only showing top 5 rows
Records with invalid discount_percent (< 0 or > 100): 8
+--------+----------------+----------------+
|order_id|         product|discount_percent|
+--

### 2.5 Inspection: Inconsistent and Invalid Dates
Inspect dates that do not conform to standard `yyyy-MM-dd` or contain impossible calendar dates (such as `31/02/2026`, `2025-02-30`, `2026-13-05`, `not-a-date`).


In [12]:
# Strict calendar date validation function
def validate_date(date_str):
    if not date_str:
        return "MISSING"
    date_str = str(date_str).strip()
    valid_formats = ["%Y-%m-%d", "%Y/%m/%d", "%m-%d-%Y", "%d-%m-%Y", "%d/%m/%Y", "%m/%d/%Y"]
    for fmt in valid_formats:
        try:
            dt = datetime.strptime(date_str, fmt)
            return dt.strftime("%Y-%m-%d")
        except (ValueError, TypeError):
            continue
    return "INVALID"

validate_date_udf = F.udf(validate_date, StringType())

# Inspect non-standard date records
date_inspection_df = raw_df.withColumn("parsed_status", validate_date_udf(F.col("order_date")))

non_standard_dates = date_inspection_df.filter(F.col("order_date") != F.col("parsed_status"))
print(f"Records with non-standard or invalid dates: {non_standard_dates.count()}")
non_standard_dates.select("order_id", "order_date", "parsed_status").show(truncate=False)


Records with non-standard or invalid dates: 6
+--------+----------+-------------+
|order_id|order_date|parsed_status|
+--------+----------+-------------+
|100097  |31/02/2026|INVALID      |
|100358  |2025/07/14|2025-07-14   |
|100448  |not-a-date|INVALID      |
|100634  |2025-02-30|INVALID      |
|100637  |2026-13-05|INVALID      |
|100969  |09-15-2026|2026-09-15   |
+--------+----------+-------------+



## 3. Basic Preprocessing - Silver Layer
Execute the 6 required preprocessing operations:
1. **Remove exact duplicate rows** (`dropDuplicates()`).
2. **Handle missing values**: Impute missing `customer_name` from other transactions of the same customer; fill remaining missing values with `"Unknown"`.
3. **Trim extra spaces and standardize capitalization**: Apply `trim()` and `initcap()` to text fields.
4. **Validate numerical fields**: Remove records with quantity $\le 0$, unit price $\le 0$, or discount percent outside $[0, 100]$.
5. **Standardize dates and handle invalid dates**: Convert valid dates to ISO `yyyy-MM-dd` format; discard unparseable / impossible calendar dates.
6. **Preserve useful records**: Keep valid edge cases (e.g., zero discount, cancelled / returned orders, high value orders).


In [13]:
# Step 1: Remove exact duplicate rows
step1_dedup_df = raw_df.dropDuplicates()
dedup_count = step1_dedup_df.count()
print(f"Step 1 - After removing duplicates: {dedup_count} rows (dropped {total_raw_count - dedup_count} duplicates)")


Step 1 - After removing duplicates: 990 rows (dropped 10 duplicates)


In [14]:
# Step 2 & 3: Handle missing values, trim extra spaces, and standardize capitalization

# Trim all text columns
trimmed_df = step1_dedup_df
for c in text_cols:
    trimmed_df = trimmed_df.withColumn(c, F.trim(F.col(c)))

# Standardize capitalization using initcap
standardized_text_df = trimmed_df \
    .withColumn("category", F.initcap(F.col("category"))) \
    .withColumn("city", F.initcap(F.col("city"))) \
    .withColumn("state", F.initcap(F.col("state"))) \
    .withColumn("order_status", F.initcap(F.col("order_status")))

# Impute missing customer_name using existing name for the same customer_id via Window function
window_spec = Window.partitionBy("customer_id")
imputed_name_col = F.first(
    F.when((F.col("customer_name").isNotNull()) & (F.col("customer_name") != ""), F.col("customer_name")),
    ignorenulls=True
).over(window_spec)

clean_missing_df = standardized_text_df \
    .withColumn("customer_name", F.coalesce(
        F.when(F.col("customer_name") != "", F.col("customer_name")),
        imputed_name_col,
        F.lit("Unknown")
    )) \
    .withColumn("customer_name", F.initcap(F.col("customer_name"))) \
    .withColumn("payment_method", F.when(
        (F.col("payment_method").isNull()) | (F.col("payment_method") == ""),
        F.lit("Unknown")
    ).otherwise(F.col("payment_method"))) \
    .withColumn("city", F.when(
        (F.col("city").isNull()) | (F.col("city") == ""),
        F.lit("Unknown")
    ).otherwise(F.col("city")))

print("Step 2 & 3 - Handled missing values, trimmed whitespace, and standardized capitalization.")


Step 2 & 3 - Handled missing values, trimmed whitespace, and standardized capitalization.


In [15]:
# Step 4: Apply basic validation to quantity, unit price, and discount percentage
# Filter out records that are clearly corrupt / invalid
valid_numerical_df = clean_missing_df.filter(
    (F.col("quantity") > 0) &
    (F.col("unit_price") > 0) &
    (F.col("discount_percent") >= 0) &
    (F.col("discount_percent") <= 100)
)

print(f"Step 4 - After numerical validation: {valid_numerical_df.count()} rows (dropped {dedup_count - valid_numerical_df.count()} rows with invalid numbers)")


Step 4 - After numerical validation: 962 rows (dropped 28 rows with invalid numbers)


In [16]:
# Step 5 & 6: Convert valid dates to one consistent format (yyyy-MM-dd) and handle clearly invalid dates
standardize_date_clean_udf = F.udf(
    lambda d: validate_date(d) if validate_date(d) != "INVALID" and validate_date(d) != "MISSING" else None,
    StringType()
)

silver_df = valid_numerical_df \
    .withColumn("order_date_cleaned", standardize_date_clean_udf(F.col("order_date"))) \
    .filter(F.col("order_date_cleaned").isNotNull()) \
    .withColumn("order_date", F.to_date(F.col("order_date_cleaned"), "yyyy-MM-dd")) \
    .drop("order_date_cleaned")

silver_count = silver_df.count()
print(f"Step 5 & 6 - Final Silver Layer record count: {silver_count:,} rows (dropped {valid_numerical_df.count() - silver_count} invalid date rows)")


Step 5 & 6 - Final Silver Layer record count: 958 rows (dropped 4 invalid date rows)


In [17]:
# Save preprocessed DataFrame inside Silver folder
# 1. Save distributed output in data_lake/silver/preprocessed_sales/
silver_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(silver_dir)

# 2. Also output consolidated submission CSV to silver/sales_silver.csv
silver_df.coalesce(1).write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("temp_silver_csv")

temp_csv_file = [f for f in os.listdir("temp_silver_csv") if f.startswith("part-") and f.endswith(".csv")][0]
shutil.copy(os.path.join("temp_silver_csv", temp_csv_file), "silver/sales_silver.csv")
shutil.rmtree("temp_silver_csv")

print("Successfully saved preprocessed Silver dataset to:")
print(f"  1. {silver_dir}")
print(f"  2. silver/sales_silver.csv")


Successfully saved preprocessed Silver dataset to:
  1. data_lake/silver/preprocessed_sales
  2. silver/sales_silver.csv


## 4. Verification
- Verify the row count before and after preprocessing.
- Read the Silver data back with Spark and display a few rows.
- Briefly summarize which preprocessing operations were performed.


In [18]:
# Row count before and after preprocessing
print("==================================================")
print(f"Row count BEFORE preprocessing (Bronze): {total_raw_count:,}")
print(f"Row count AFTER preprocessing (Silver) : {silver_count:,}")
print(f"Total filtered / invalid records dropped : {total_raw_count - silver_count:,}")
print("==================================================")


Row count BEFORE preprocessing (Bronze): 1,000
Row count AFTER preprocessing (Silver) : 958
Total filtered / invalid records dropped : 42


In [19]:
# Read the Silver data back with Spark and inspect
silver_readback_df = spark.read.csv(
    silver_dir,
    header=True,
    inferSchema=True
)

print("=== Silver Layer Schema ===")
silver_readback_df.printSchema()

print("=== First 5 Rows of Silver Layer ===")
silver_readback_df.show(5, truncate=False)

# Verify zero nulls remain in critical fields
print("=== Null Value Check on Silver Layer ===")
silver_readback_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_readback_df.columns
]).show(vertical=True)


=== Silver Layer Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- order_status: string (nullable = true)

=== First 5 Rows of Silver Layer ===
+--------+-----------+-------------+-----------+--------+--------+----------+----------------+----------------+----------+--------------+----------+------------+
|order_id|customer_id|customer_name|product    |category|quantity|unit_price|discount_percent|payment_method  |city      |state         |order_date|order_status|
+--------+-----------+-------------+-----------+--------+--

### 4.1 Summary of Preprocessing Operations Performed
| Step | Operation | Method & Spark Function | Rows Affected / Impact |
|---|---|---|---|
| **1** | **Exact Duplicate Removal** | `raw_df.dropDuplicates()` | 10 duplicate rows dropped (1,000 $\rightarrow$ 990 rows) |
| **2** | **Missing Value Imputation** | Window function `partitionBy("customer_id")` + fallback to `"Unknown"` | Imputed 10 customer names; imputed missing payment methods & cities |
| **3** | **Text Standardization & Trimming** | `F.trim()` and `F.initcap()` on string fields | Removed extra spaces; unified mixed casing (e.g. `SPORTS` $\rightarrow$ `Sports`, `DELHI` $\rightarrow$ `Delhi`) |
| **4** | **Numerical Validation** | Filtered: `quantity > 0`, `unit_price > 0`, `0 <= discount <= 100` | 28 corrupt rows removed (12 qty $\le 0$, 8 price $\le 0$, 8 invalid discount) |
| **5** | **Date Validation & Standardization** | Multi-format strict date parsing + filtered invalid calendar dates | 2 valid non-standard dates converted to `yyyy-MM-dd`; 4 invalid dates dropped (`31/02/2026`, `2025-02-30`, `2026-13-05`, `not-a-date`) |
| **6** | **Preserve Useful Records** | Retained legitimate business edge cases (discounts = 0%, cancelled / returned items) | Kept all 958 high-utility valid business transactions |

**Final Output**:
- Bronze Layer: `data_lake/bronze/sales.csv` (1,000 rows, raw)
- Silver Layer: `data_lake/silver/preprocessed_sales/` (958 rows, preprocessed)
- Consolidated File: `silver/sales_silver.csv` (958 rows)
